# Ejercicio 5 — Programación Orientada a Objetos
## Catálogo de Sismos USGS — Examen Final (Variante A)

**Nombre:** ___________________________  
**Carnet:** ___________________________

---

Este notebook usa las clases que implementaste en `poo_sismos.py`.

**Requisito previo:** haber completado todos los métodos de `poo_sismos.py`.

**Entrega:**
- `poo_sismos.py` con tu implementación
- Este notebook ejecutado con todos los outputs visibles

> **Ruta esperada en tu repositorio:**  
> `examenes/python/examen_final/poo_sismos.py`  
> `examenes/python/examen_final/examen_final_poo_a.ipynb`

In [ ]:
import pandas as pd

URL_SISMOS = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query"
    "?format=csv"
    "&starttime=2024-01-01"
    "&endtime=2024-06-30"
    "&minmagnitude=5.5"
)

sismos = pd.read_csv(URL_SISMOS)
print(f"Dataset cargado: {len(sismos)} sismos")
sismos.head(3)

In [ ]:
# Reutiliza la función clasificar_sismo() del Ejercicio 3 para agregar la columna 'categoria'
def clasificar_sismo(mag):
    if mag < 6.0:
        return 'Moderado-Fuerte'
    elif mag < 7.0:
        return 'Fuerte'
    elif mag < 8.0:
        return 'Mayor'
    else:
        return 'Gran terremoto'

sismos['categoria'] = sismos['mag'].apply(clasificar_sismo)
print("Columna 'categoria' agregada.")
print(sismos['categoria'].value_counts())

In [ ]:
from poo_sismos import Sismo, CatalogoSismos
print("Clases importadas correctamente.")

---
## Parte 1 — Probando la clase `Sismo` (objetos individuales)

El siguiente bloque crea tres objetos `Sismo` representativos y llama a todos sus métodos.  
Verifica que el output tiene sentido con los datos.

In [ ]:
# Tres sismos representativos del DataFrame
fila_primero  = sismos.iloc[0]
fila_mayor    = sismos.loc[sismos['mag'].idxmax()]
fila_profundo = sismos.loc[sismos['depth'].idxmax()]

def crear_sismo(fila):
    return Sismo(fila['place'], fila['time'], fila['mag'], fila['depth'], fila['magType'])

s_primero  = crear_sismo(fila_primero)
s_mayor    = crear_sismo(fila_mayor)
s_profundo = crear_sismo(fila_profundo)

for etiqueta, s in [("Primero del dataset", s_primero),
                    ("Mayor magnitud",       s_mayor),
                    ("Mayor profundidad",    s_profundo)]:
    print(f"=== {etiqueta} ===")
    print(f"  {s}")
    print(f"  clasificar()            → {s.clasificar()}")
    print(f"  clasificar_profundidad()→ {s.clasificar_profundidad()}")
    print(f"  es_peligroso()          → {s.es_peligroso()}")
    print()

### Tu turno — elige un sismo

Crea un objeto `Sismo` usando **cualquier fila** del DataFrame que tú elijas  
(puede ser el más reciente, uno de Guatemala, el más superficial, etc.).  
Muestra su `descripcion()` y justifica tu elección en un comentario.

In [ ]:
# Elige una fila y crea tu propio objeto Sismo
# Hint: sismos.loc[indice] o sismos[sismos['place'].str.contains('Guatemala')]

# TU CÓDIGO AQUÍ
mi_fila  = ...
mi_sismo = Sismo(...)

print(mi_sismo)
print(f"¿Es peligroso? {mi_sismo.es_peligroso()}")

---
## Parte 2 — Construyendo el `CatalogoSismos` con todos los datos

Ahora crea el catálogo con **todos** los sismos del DataFrame (no solo 50) y ejerce los métodos de consulta.

In [ ]:
catalogo = CatalogoSismos("Sismos Globales — Semestre 1 de 2024")

for indice, fila in sismos.iterrows():
    catalogo.agregar(Sismo(
        lugar       = fila['place'],
        fecha       = fila['time'],
        magnitud    = fila['mag'],
        profundidad = fila['depth'],
        tipo_escala = fila['magType'],
    ))

print(f"Catálogo construido: {len(catalogo)} sismos")

In [ ]:
# ── el_mas_intenso() ────────────────────────────────────────────────────────
mas_intenso = catalogo.el_mas_intenso()
print("Sismo más intenso del catálogo completo:")
print(" ", mas_intenso)
print(f"  ¿Es peligroso? → {mas_intenso.es_peligroso()}")

In [ ]:
# ── filtrar_por_categoria() ──────────────────────────────────────────────────
print("Sismos por categoría (catálogo completo):")
for cat in ['Moderado-Fuerte', 'Fuerte', 'Mayor', 'Gran terremoto']:
    grupo = catalogo.filtrar_por_categoria(cat)
    print(f"  {cat:<18}: {len(grupo)} sismos")
    # Muestra los primeros 2 de cada categoría
    for s in grupo[:2]:
        print(f"      {s}")

In [ ]:
# ── resumen() ────────────────────────────────────────────────────────────────
catalogo.resumen()

### Tu turno — sismos peligrosos

Usa `filtrar_por_categoria()` y el método `es_peligroso()` para  
**listar todos los sismos que son peligrosos** (Mayor o Gran terremoto, superficiales).  
Muestra cuántos hay y su `descripcion()`.

In [ ]:
# Hint: usa filtrar_por_categoria() para obtener 'Mayor' y 'Gran terremoto'
#       luego filtra los que es_peligroso() == True con un for

# TU CÓDIGO AQUÍ

---
## Parte 3 — Verificación automática

Esta celda confirma que tu `clasificar()` produce los mismos resultados  
que la función `clasificar_sismo()` del Ejercicio 3 para **todos** los sismos.

In [ ]:
errores = 0
for _, fila in sismos.iterrows():
    s = Sismo("", "", fila['mag'], fila['depth'])
    if s.clasificar() != fila['categoria']:
        errores += 1
        print(f"  ✗ mag={fila['mag']:.2f}  pandas={fila['categoria']}  poo={s.clasificar()}")

if errores == 0:
    print(f"✓ Todos los {len(sismos)} sismos clasifican correctamente.")
else:
    print(f"✗ {errores} sismos con clasificación incorrecta. Revisa tu método clasificar().")

---
## Preguntas de interpretación

1. ¿Cuántos sismos peligrosos (`es_peligroso() == True`) encontraste en el catálogo completo?  
   ¿En qué regiones del mundo ocurrieron?

   *Tu respuesta:* ___

2. ¿Cuántos sismos de categoría `'Gran terremoto'` (mag ≥ 8.0) hay en el semestre?  
   Si no hay ninguno, ¿por qué tiene sentido dada la frecuencia global de ese tipo de evento?

   *Tu respuesta:* ___

3. ¿Qué ventaja tiene usar la clase `CatalogoSismos` frente a trabajar directamente  
   con el DataFrame de pandas para este tipo de consultas?

   *Tu respuesta:* ___